# Basic LLM Training
Train a GPT-like Large Language Model from scratch on `the-verdict.txt`.

This notebook supports execution both in **Google Colab** (via VS Code or Web) and in a **local environment**.

In [ ]:
# 1. Environment Setup & Dependency Installation
%pip install -q torch tiktoken

import sys
import os

# Detect if running in Google Colab (remote VM)
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    print("Running in Google Colab environment.")
    repo_dir = "/content/basic-llm"
    if not os.path.exists(repo_dir):
        print("Cloning repository into Colab...")
        !git clone https://github.com/Vishaal-sathya/basic-llm.git {repo_dir}
    else:
        print("Pulling latest updates into Colab...")
        !git -C {repo_dir} pull

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    %cd {repo_dir}
else:
    print("Running in local environment.")
    repo_dir = os.path.abspath(".")
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Python sys.path[0]: {sys.path[0]}")

In [ ]:
# 2. Import modules
import torch
import tiktoken
from llm.llm_config import LLM_CONFIG
from llm.llm_architecture import LLMModel
from training.prepare_data import create_loaders
from training.train_model import train_model

print("All modules imported successfully!")
print("LLM Configuration:", LLM_CONFIG)

In [ ]:
# 3. Create training and validation data loaders
file_path = "the-verdict.txt"
train_ratio = 0.9

# max_length=256 ensures enough tokens in the-verdict.txt validation set (~534 tokens)
train_loader, val_loader = create_loaders(
    file_path=file_path,
    train_ratio=train_ratio,
    batch_size=2,
    max_length=256,
    stride=256
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# 4. Initialize model on GPU (if available) or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(123)
model = LLMModel(LLM_CONFIG)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
tokenizer = tiktoken.get_encoding('gpt2')

In [ ]:
# 5. Train the model
num_epochs = 10

train_losses, val_losses, tokens_seen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=1,
    start_context="Every effort moves you",
    tokenizer=tokenizer
)